In [30]:
import csv

# Mapping tags to integers
tag_map = {
    'O': 0,
    'B-ACTOR': 1,
    'I-ACTOR': 2,
    'B-USE_CASE': 3,
    'I-USE_CASE': 4
}

def parse_input_file(filepath):
    with open(filepath, 'r', encoding='utf-8') as file:
        content = file.read().strip()
    
    sentences = content.split('\n\n')
    parsed_data = []

    for sentence in sentences:
        tokens = []
        tags = []

        for line in sentence.strip().split('\n'):
            if not line.strip():
                continue
            splitted = line.rsplit('-')
            if splitted[:-1] == 'O':
                tags.append(0)
            else:
                b_or_i = splitted[-2]
                ent =  splitted[-1]
                raw_tag = b_or_i + "-" + ent
                label = tag_map.get(raw_tag, 0)
                tags.append(label)
            tokens.append(splitted[0])

        parsed_data.append((tokens, tags))
    
    return parsed_data

def write_to_csv(data, output_file):
    with open(output_file, 'w', newline='', encoding='utf-8') as csvfile:
        writer = csv.writer(csvfile)
        writer.writerow(['tokens', 'tags'])

        for tokens, tags in data:
            writer.writerow([str(tokens), str(tags)])

# Example usage
input_path = '../data/usecase/dataset-raymond-iob.txt'     # Replace with your input file
output_path = '../data/usecase/dataset_transformed_new.csv'   # Desired output file

parsed = parse_input_file(input_path)
write_to_csv(parsed, output_path)


# separate it into train-val-test

In [32]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split

# Read the dataset
df = pd.read_csv(output_path)

# Check the content of the dataset
print(f"Dataset shape: {df.shape}")
print(df.head())

# Extract the training set (from 0 to 1670) and test set (the rest)
train_df = df[:1669]
test_df = df[1669:]

print(f"Training set shape: {train_df.shape}")
print(f"Test set shape: {test_df.shape}")

# Further split the training set into train and validation sets (80%/20%)
train_data, val_data = train_test_split(train_df, test_size=0.2, random_state=42)

print(f"Final train set shape: {train_data.shape}")
print(f"Validation set shape: {val_data.shape}")

# Save the datasets
output_dir = os.path.dirname(output_path)
train_output_path = os.path.join(output_dir, 'usecase-train-hf.csv')
val_output_path = os.path.join(output_dir, 'usecase-val-hf.csv')
test_output_path = os.path.join(output_dir, 'usecase-test-hf.csv')

train_data.to_csv(train_output_path, index=False)
val_data.to_csv(val_output_path, index=False)
test_df.to_csv(test_output_path, index=False)

print(f"Training data saved to: {train_output_path}")
print(f"Validation data saved to: {val_output_path}")
print(f"Test data saved to: {test_output_path}")

Dataset shape: (1797, 2)
                                              tokens  \
0  ['The', 'Functional', 'Requirements', 'Specifi...   
1  ['CCTNS', 'V1.0', 'functionality', 'is', 'desi...   
2  ['The', 'functionality', 'of', 'the', 'CCTNS',...   
3  ['Citizens', 'can', 'register', 'their', 'comp...   
4  ['After', 'a', 'complaint', 'is', 'initiated',...   

                                                tags  
0  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
1  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
2  [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
3  [1, 0, 3, 4, 4, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...  
4  [0, 0, 0, 0, 0, 0, 1, 0, 0, 3, 0, 0, 0, 0, 0, ...  
Training set shape: (1669, 2)
Test set shape: (128, 2)
Final train set shape: (1335, 2)
Validation set shape: (334, 2)
Training data saved to: ../data/usecase\usecase-train-hf.csv
Validation data saved to: ../data/usecase\usecase-val-hf.csv
Test data saved to: ../data/usecase\usecase-test-hf.csv
